# Evaluating Language Model Outputs

Large Language Models (LLMs) generate rich, human‑like text—yet assessing the quality of that text remains a complex challenge. In this notebook, we experiment with several complementary evaluation methods to compare how well generated responses align with reference outputs and task expectations.

In [ ]:
import nltk                                      # Natural Language Toolkit for NLP tasks like tokenization
nltk.download('punkt_tab')                       # Downloads Punkt tokenizer models for sentence splitting
from rouge_score import rouge_scorer             # ROUGE scorer for evaluating text generation quality
import openai                                    # OpenAI API client for GPT models and embeddings
import numpy as np                               # Numerical computing library for arrays and math operations
from typing import List, Dict                    # Type hints for list and dictionary annotations

Let's start with some data to explore various evaluations methods for LLM applications.  In the data below, we prompt a model with the same question, but obtain different generated responses.  

In [ ]:
data = [
    {
        "prompt": "Who wrote the novel 'Pride and Prejudice'?",
        "reference": "Jane Austen wrote the novel 'Pride and Prejudice.'",
        "generated": "The novel 'Pride and Prejudice' was written by Jane Austen."
    },
    {
        "prompt": "Who wrote the novel 'Pride and Prejudice'?",
        "reference": "Jane Austen wrote the novel 'Pride and Prejudice.'",
        "generated": "It was penned by the English author Jane Austen."
    },
    {
        "prompt": "Who wrote the novel 'Pride and Prejudice'?",
        "reference": "Jane Austen wrote the novel 'Pride and Prejudice.'",
        "generated": "The author of 'Pride and Prejudice' is Jane Austen."
    }
]

## Exact match 

Exact match is a strict metric often used for tasks with a single correct answer (like multiple-choice or closed-form responses), but it fails to capture semantically correct variations that differ in phrasing or formatting.

In [ ]:
## 1. Simple Solutions (Exact Match, Multiple Choice Style)

def exact_match(reference: str, generated: str) -> bool:
    """Closed-form: exact string match"""
    return reference.lower().strip() == generated.lower().strip()

# Demo
for i, item in enumerate(data):
    ref = item["reference"]
    gen = item["generated"]
    print(f"Example {i+1}: \n prompt:{item['prompt']} \n reference:{item['reference']} \n generated {item['generated']} \n Exact Match = {exact_match(ref, gen)}")
# Output: False for all - shows limitation of exact matching

## Automated Metrics: BLEU and ROUGE 

These automated metrics evaluate how closely a model’s response aligns with reference text by measuring surface‑level word overlap using BLEU and ROUGE. While they reward similar wording, they often can penalize creative or paraphrased outputs, and are less reliable for open‑ended LLM tasks.

In [ ]:
## 2. Automated Metrics (BLEU, ROUGE)
def compute_bleu(reference: str, generated: str) -> float:
    """Simple BLEU using nltk"""
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smoothie = SmoothingFunction().method1
    ref_tokens = [nltk.word_tokenize(reference.lower())]
    gen_tokens = nltk.word_tokenize(generated.lower())
    return sentence_bleu(ref_tokens, gen_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie )

def compute_rouge(reference: str, generated: str) -> Dict:
    """ROUGE scores"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    return scorer.score(reference, generated)

# Demo table
print("\nAutomated Metrics:")
metrics_results = []
for item in data:
    bleu = compute_bleu(item["reference"], item["generated"])
    rouge = compute_rouge(item["reference"], item["generated"])
    metrics_results.append({
        "Example": item["prompt"][:30] + "...",
        "BLEU": f"{bleu:.3f}",
        "ROUGE-1": f"{rouge['rouge1'].fmeasure:.3f}",
        "ROUGE-L": f"{rouge['rougeL'].fmeasure:.3f}"
    })
    print(f"Example {i+1}: \n prompt:{item['prompt']} \n reference:{item['reference']} \n generated {item['generated']} \n BLEU={bleu:.3f}, ROUGE1={rouge['rouge1'].fmeasure:.3f}")

# These catch semantic similarity better than exact match!

## LLM as a judge 

This approach uses an LLM itself as an evaluator, asking it to judge model outputs either as “GOOD/BAD” or on a 1–5 scale for quality dimensions like accuracy and clarity. It offers more semantic and context‑aware assessments than automated metrics but can introduce bias, variability across models, and depend on prompt design. Binary GOOD/BAD judgments are generally more reliable and stable, while 1–5 Likert scores can be noisier but provide finer‑grained differentiation when carefully calibrated.  Typically you will want to evaluate the evaluator when using LLM as a judge. 

In [ ]:
import requests
import json

# Ollama runs locally at http://localhost:11434 by default
OLLAMA_URL = "http://localhost:11434/api/generate"

def llm_judge_binary(prompt: str, reference: str, generated: str) -> str:
    """Binary judgment: good/bad using local Ollama"""
    ollama_payload = {
        "model": "gpt-oss:20b",  # or qwen2.5:7b, gemma3:4b
        "prompt": f"""You are an expert evaluator. Respond only with 'GOOD' or 'BAD'.

                      Prompt: {prompt}
                      Reference: {reference}
                      Generated: {generated}
                        
                      Is the generated answer accurate, complete, and helpful compared to reference? Answer only: GOOD or BAD""",
        "stream": False,
        "temperature": 0.0,
        "options": {
            "temperature": 0.0,
            "top_p": 0.1
        }
    }
    
    response = requests.post(OLLAMA_URL, json=ollama_payload)
    if response.status_code == 200:
        result = response.json()
        return result["response"].strip()
    return "ERROR"

def llm_judge_likert(prompt: str, reference: str, generated: str) -> int:
    """1-5 Likert scale using local Ollama"""
    ollama_payload = {
        "model": "gpt-oss:20b",
        "prompt": f"""Rate 1-5 where 1=poor, 5=excellent. Respond ONLY with number.

Prompt: {prompt}
Reference: {reference}
Generated: {generated}

Score the generated response vs reference (accuracy, completeness, clarity): 1-5? Answer ONLY with number.""",
        "stream": False,
        "temperature": 0.0,
        "options": {
            "temperature": 0.0,
            "top_p": 0.1
        }
    }
    
    response = requests.post(OLLAMA_URL, json=ollama_payload)
    if response.status_code == 200:
        result = response.json()
        try:
            return int(result["response"].strip())
        except:
            return 0
    return 0

# Demo (same as before)
print("\nLLM-as-a-Judge (Local Ollama):")
judge_results = []
for item in data:
    binary = llm_judge_binary(item["prompt"], item["reference"], item["generated"])
    likert = llm_judge_likert(item["prompt"], item["reference"], item["generated"])
    judge_results.append({
        "Example": item["prompt"][:30] + "...",
        "Binary": binary,
        "Likert (1-5)": likert
    })
    print(f"Example {i+1}: \n prompt:{item['prompt']} \n reference:{item['reference']} \n generated {item['generated']} \n binary: {binary}, 5 point scale: {likert}/5")


## Functional Scorers 

Rule‑based scorers define human‑designed heuristics—such as keyword presence, length, punctuation, or schema conformance—and convert them into numerical scores, making them fast, interpretable, and easy to customize for specific tasks. In LLM evaluation, they are often used to enforce basic quality or safety requirements (for example, “must mention key facts,” “must not exceed X tokens,” or “must follow a structured format”), serving as lightweight, reference‑free checks that complement more complex metrics like automated scores or LLM‑as‑a‑judge evaluations.  

In [ ]:
from typing import List, Dict

def functional_scorer(generated: str, key_words: List[str]) -> Dict:
    """Simple rule-based: check for key terms, length, structure"""
    score = 0
    tokens = generated.lower().split()

    # True if any of the key_words appears as a token in generated
    has_key_words = any(kw.lower() in tokens for kw in key_words)

    good_length = 15 < len(tokens) < 50
    ends_with_period = generated.strip().endswith(".")

    if has_key_words:
        score += 3
    if good_length:
        score += 1
    if ends_with_period:
        score += 1

    return {
        "functional_score": score,
        "max": 5,
        "details": f"keywords:{has_key_words}, len:{good_length}, punct:{ends_with_period}"
    }

# Demo
for item in data:
    func = functional_scorer(item["generated"],['Jane', 'Austen'])
    print(f"Example {i+1}: \n prompt:{item['prompt']} \n reference:{item['reference']} \n generated {item['generated']} \n {func}")

Finally, lets put all these metrics together in one function so that we can view and compare all results

In [ ]:
## 5. Evaluation Summary Table

import pandas as pd

def evaluate_dataset_all_metrics(data,key_words):
    all_results = []
    for i, item in enumerate(data):
        all_results.append({
            "Prompt": item["prompt"][:40] + "...",
            "generated": item["reference"],
            "reference": item["generated"],
            "Exact Match": exact_match(item["reference"], item["generated"]),
            "BLEU": compute_bleu(item["reference"], item["generated"]),
            "ROUGE-L": compute_rouge(item["reference"], item["generated"])['rougeL'].fmeasure,
            "LLM Binary": llm_judge_binary(item["prompt"], item["reference"], item["generated"]),
            "LLM Likert": llm_judge_likert(item["prompt"], item["reference"], item["generated"]),
            "Functional": functional_scorer(item["generated"],key_words)["functional_score"]
        })
    
    df = pd.DataFrame(all_results)
    
    return df

evaluate_dataset_all_metrics(data,['Jane','Austen'])

In [ ]:
# Key Takeaways:
# - Exact match fails for paraphrasing 
# - BLEU/ROUGE capture n-gram overlap but miss deep semantics 
# - LLM-as-judge excels at binary but gets fuzzy on scales 
# - Functional scorers are cheap/fast but domain-specific

# Exercise

Ok, we have a few additional datasets where we could compare different metrics accross various response from the LLM.  Let's see how the various metrics do in differenct domains! For each example, you will need to specify a list of key words for the functional scorer to use. 

In [ ]:
reasoning_data = [
    {
        "prompt": "A store sells apples at 3 dollars each or 10 dollars for 4. What’s the cheaper option per apple?",
        "reference": "The 4-for-10 deal is cheaper, because each apple costs 2.50 instead of 3.00.",
        "generated": "Buying four apples for ten dollars is cheaper, since that makes them 2.50 each rather than 3 dollars."
    },
    {
        "prompt": "A store sells apples at 3 dollars each or 10 dollars for 4. What’s the cheaper option per apple?",
        "reference": "The 4-for-10 deal is cheaper, because each apple costs 2.50 instead of 3.00.",
        "generated": "The 4-for-10 bundle is the better deal; it works out to 2.50 per apple, while a single apple is 3 dollars."
    },
    # “Almost right” answer
    {
        "prompt": "A store sells apples at 3 dollars each or 10 dollars for 4. What’s the cheaper option per apple?",
        "reference": "The 4-for-10 deal is cheaper, because each apple costs 2.50 instead of 3.00.",
        "generated": "They cost about the same, so there isn’t really a cheaper option."
    }
]


In [ ]:
evaluate_dataset_all_metrics(reasoning_data,['cheaper'])

In [ ]:
creative_data = [
    {
        "prompt": "Write a one-sentence slogan for a coffee shop that stays open all night.",
        "reference": "Awake all night, brewing your perfect cup.",
        "generated": "Your cozy cup of energy, open all night long."
    },
    {
        "prompt": "Write a one-sentence slogan for a coffee shop that stays open all night.",
        "reference": "Awake all night, brewing your perfect cup.",
        "generated": "Coffee that never sleeps, just like you."
    },
    {
        "prompt": "Write a one-sentence slogan for a coffee shop that stays open all night.",
        "reference": "Awake all night, brewing your perfect cup.",
        "generated": "Open all night, keeping your ideas and your coffee flowing."
    }
]

In [ ]:
evaluate_dataset_all_metrics(creative_data,['night','coffee'])

In [ ]:
summarization_data = [
    {
        "prompt": (
            "Summarize: Our app just launched a new offline mode that lets users "
            "save articles to read without an internet connection. It also adds "
            "better search filters so people can quickly find the content they care about."
        ),
        "reference": "The app now offers offline article access and improved search filters for finding relevant content.",
        "generated": "The latest update lets users save articles to read offline and includes more powerful filters to quickly locate the stories they want."
    },
    {
        "prompt": (
            "Summarize: Our app just launched a new offline mode that lets users "
            "save articles to read without an internet connection. It also adds "
            "better search filters so people can quickly find the content they care about."
        ),
        "reference": "The app now offers offline article access and improved search filters for finding relevant content.",
        "generated": "We introduced offline reading for saved articles and upgraded search options so users can more easily find content they enjoy."
    }
]


In [ ]:
evaluate_dataset_all_metrics(summarization_data,['app','offline'])

In [ ]:
adversarial_data = [
    {
        "prompt": "If a snake is disguised as a bird with painted wings, how many legs does it have?",
        "reference": "It still has zero legs, because snakes do not have legs.",
        "generated": "It has zero legs, since painting wings on it doesn’t change the fact that snakes have no legs."
    },
    # Plausible but wrong
    {
        "prompt": "If a snake is disguised as a bird with painted wings, how many legs does it have?",
        "reference": "It still has zero legs, because snakes do not have legs.",
        "generated": "It would have two legs, because birds typically have two legs."
    }
]

In [ ]:
evaluate_dataset_all_metrics(adversarial_data,['zero'])